In [2]:
import shared_libraries.model_search as search

import pandas as pd

import pickle

In [3]:

with open("./ig_data/data_by_circuit_one_hot_encoded.pickle", "rb") as file:
    data_by_circuit = pickle.load(file)


Here, we use a diverse group of algorithms together with hyperparameter tuning to train ML models on a single circuit's data. Based on the results, we'll select the best algorithms and train them on all circuits.

In [4]:
initial_test_circuit = "Catalunya"

In [ ]:

initial_models = search.search(data_by_circuit[initial_test_circuit], "LapTimeZScore", search.DEFAULT_SEARCHES)

In [6]:
initial_models_df = pd.DataFrame(initial_models, ["GSCV"]).T
initial_models_df["Score"] = initial_models_df["GSCV"].apply(lambda cv: cv.best_score_)
initial_models_df.sort_values(by="Score", ascending=False, inplace=True)
initial_models_df

,GSCV,Score
XGBRegressor,GridSearchCV(estimator=XGBRegressor(base_score...,0.661179
GradientBoostingRegressor,GridSearchCV(estimator=GradientBoostingRegress...,0.632489
XGBRFRegressor,GridSearchCV(estimator=XGBRFRegressor(base_sco...,0.623613
ExtraTreesRegressor,GridSearchCV(estimator=ExtraTreesRegressor(n_j...,0.621813
RandomForestRegressor,GridSearchCV(estimator=RandomForestRegressor(n...,0.581748
AdaBoostRegressor,GridSearchCV(estimator=AdaBoostRegressor(rando...,0.418851
MLPRegressor,GridSearchCV(estimator=Pipeline(steps=[('stand...,0.314548
PolynomialLassoCV,GridSearchCV(estimator=Pipeline(steps=[('stand...,0.073367
SVR_rbf,GridSearchCV(estimator=Pipeline(steps=[('stand...,0.072443
PolynomialElasticNetCV,GridSearchCV(estimator=Pipeline(steps=[('stand...,0.055124


Here we select the models the scores of which were at or past one standard deviation from the mean of all scores.

In [7]:
scores = initial_models_df["Score"]
threshold = scores.mean() + scores.std()

chosen_models = initial_models_df.loc[initial_models_df["Score"] >= threshold]


In [8]:
chosen_models_searches = {}
for key in chosen_models.index:
    chosen_models_searches[key] = search.DEFAULT_SEARCHES[key]

Finally, we train these models on the remaining circuits.

In [ ]:
models = {}
i = 1
for circuit, data in [(circuit, data) for circuit, data in data_by_circuit.items() if circuit != initial_test_circuit]:
    print(f"{i}. {circuit}")
    models[circuit] = search.search(data, "LapTimeZScore", chosen_models_searches)
    i += 1

# We also add the models trained earlier
models[initial_test_circuit] = chosen_models.drop("Score", axis="columns").T.iloc[0].to_dict()

In [10]:
import os

In [11]:
folder = "./ig_models"
os.makedirs(folder, exist_ok=True)

with open(f"{folder}/models.pickle", "wb") as file:
    pickle.dump(models, file)